In [2]:
import os
os.getcwd()

'C:\\Users\\User\\OneDrive - Universiti Kebangsaan Malaysia\\sem 2\\MACHINE LEARNING\\ASSIGNMENT2\\test code'

# PART B

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [9]:
## Load dataset
data=pd.read_csv('ASSG22025.csv')
data.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,target
0,1.286759,-3.452735,-0.595082,2.852650,5.289887,-0.366363,4.533496,0.347942,0.717730,-0.645499,5.186470,0.202211,2.479828,5.034365,1.927844,0
1,-0.445937,-4.098332,-0.678555,0.242937,4.107822,-1.645675,4.933900,-0.415859,0.107588,-1.912968,3.911879,1.950317,2.707602,6.150720,1.975771,0
2,1.654022,-1.064226,-1.480801,-0.719527,2.978319,0.482093,-1.054311,2.129813,-1.704825,0.422010,1.077011,-4.565876,0.380761,3.924259,0.596692,0
3,-2.432039,3.309322,-0.458722,-1.967034,-4.950714,0.108292,-4.921838,0.875585,-0.009648,-1.674498,-6.128834,-1.171714,-4.314178,-3.931018,-2.002449,0
4,-1.084632,1.105736,-0.208542,0.297446,-1.669344,0.609055,-2.571997,-0.038799,0.985999,-1.571114,-1.363833,-1.039746,-0.616285,-1.255084,-2.065537,0


In [11]:
X=data.drop(columns=['target'])
y=data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [13]:
###### ----- Logistic regression (baseline) ----- #####
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log = LogisticRegression(max_iter=1000)
log.fit(X_train_scaled, y_train)

y_pred_lr = log.predict(X_test_scaled)
y_prob_lr = log.predict_proba(X_test_scaled)[:, 1]

print("\n===== Logistic Regression (Baseline) =====")
print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1 Score :", f1_score(y_test, y_pred_lr))
print("AUC      :", roc_auc_score(y_test, y_prob_lr))


===== Logistic Regression (Baseline) =====
Accuracy : 0.95
Precision: 1.0
Recall   : 0.0625
F1 Score : 0.11764705882352941
AUC      : 0.777068661971831


In [15]:
# -------- Decision Tree (Baseline) --------
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)
y_prob_tree = tree.predict_proba(X_test)[:, 1]

print("\n===== Decision Tree (Baseline) =====")
print("Accuracy :", accuracy_score(y_test, y_pred_tree))
print("Precision:", precision_score(y_test, y_pred_tree, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_tree))
print("F1 Score :", f1_score(y_test, y_pred_tree))
print("AUC      :", roc_auc_score(y_test, y_prob_tree))


===== Decision Tree (Baseline) =====
Accuracy : 0.9366666666666666
Precision: 0.4
Recall   : 0.375
F1 Score : 0.3870967741935484
AUC      : 0.6716549295774649


+ Baseline models trained without imbalance handling appear to perform well based on accuracy, but this is misleading due to the highly skewed class distribution.

#### Logistic regression
+ Accuracy = 0.95 (very high)
+ Precision = 1.00
+ Recall = ❌ 0.0625 (almost zero)
+ F1-score = 0.118
+ Model correctly predicts almost all majority-class samples but fails to detect most minority-class cases
+ That gives high accuracy, but misses 94% of  the rare events
+ Model looks good on paper but perform poorly in reality

#### Decision tree
+ Accuracy = 0.936
+ Precision = 0.4
+ Recall = 0.375 (much better than LR)
+ F1-score = 0.387
+ Even though its accuracy is lower, it detects more minority cases, meaning it is actually more useful than logistic regression.

# PART C

### Logistic Regression (Class-Weight)

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Logistic Regression with class weights
lr_classweight = LogisticRegression(class_weight='balanced', max_iter=1000)
lr_classweight.fit(X_train, y_train)
y_pred = lr_classweight.predict(X_test)
y_proba = lr_classweight.predict_proba(X_test)[:, 1]

# Metrics
print("=== Logistic Regression (Class-Weight) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))


=== Logistic Regression (Class-Weight) ===
Accuracy: 0.7666666666666667
Precision: 0.13513513513513514
Recall: 0.625
F1 Score: 0.2222222222222222
AUC: 0.7731073943661971


### Logistic Regression (Random Oversampling)

In [25]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split

# Apply Random Oversampling
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

# Train Logistic Regression with resampled data
lr_oversampled = LogisticRegression(class_weight='balanced', max_iter=1000)
lr_oversampled.fit(X_resampled, y_resampled)
y_pred_oversampled = lr_oversampled.predict(X_test)
y_proba_oversampled = lr_oversampled.predict_proba(X_test)[:, 1]

# Metrics after Random Oversampling
print("=== Logistic Regression (Random Oversampling) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_oversampled))
print("Precision:", precision_score(y_test, y_pred_oversampled))
print("Recall:", recall_score(y_test, y_pred_oversampled))
print("F1 Score:", f1_score(y_test, y_pred_oversampled))
print("AUC:", roc_auc_score(y_test, y_proba_oversampled))

=== Logistic Regression (Random Oversampling) ===
Accuracy: 0.78
Precision: 0.14285714285714285
Recall: 0.625
F1 Score: 0.23255813953488372
AUC: 0.771786971830986


### Logistic Regression (Random Undersampling)

In [28]:
from imblearn.under_sampling import RandomUnderSampler

# Apply Random Undersampling
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

# Train Logistic Regression with resampled data
lr_undersampled = LogisticRegression(class_weight='balanced', max_iter=1000)
lr_undersampled.fit(X_resampled, y_resampled)
y_pred_undersampled = lr_undersampled.predict(X_test)
y_proba_undersampled = lr_undersampled.predict_proba(X_test)[:, 1]

# Metrics after Random Undersampling
print("=== Logistic Regression (Random Undersampling) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_undersampled))
print("Precision:", precision_score(y_test, y_pred_undersampled))
print("Recall:", recall_score(y_test, y_pred_undersampled))
print("F1 Score:", f1_score(y_test, y_pred_undersampled))
print("AUC:", roc_auc_score(y_test, y_proba_undersampled))

=== Logistic Regression (Random Undersampling) ===
Accuracy: 0.69
Precision: 0.1111111111111111
Recall: 0.6875
F1 Score: 0.19130434782608696
AUC: 0.761443661971831


### Logistic Regression (SMOTE)

In [41]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Check new class distribution
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# Train Logistic Regression with SMOTE
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)

# Predictions
y_pred_smote = lr_smote.predict(X_test)
y_prob_smote = lr_smote.predict_proba(X_test)[:, 1]

# Evaluation Metrics after SMOTE
print("=== Logistic Regression (SMOTE) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall:", recall_score(y_test, y_pred_smote))
print("F1 Score:", f1_score(y_test, y_pred_smote))
print("AUC:", roc_auc_score(y_test, y_prob_smote))

Before SMOTE:
target
0    1136
1      64
Name: count, dtype: int64

After SMOTE:
target
0    1136
1    1136
Name: count, dtype: int64
=== Logistic Regression (SMOTE) ===
Accuracy: 0.7666666666666667
Precision: 0.13513513513513514
Recall: 0.625
F1 Score: 0.2222222222222222
AUC: 0.768705985915493


### Logistic Regression (Threshold Moving)

In [34]:
import numpy as np

# Predicted probabilities (NumPy array)
y_prob = log.predict_proba(X_test_scaled)[:, 1]

# Moving Threshold 
window_size = 30
moving_threshold = pd.Series(y_prob).rolling(
    window=window_size, center=True
).mean().to_numpy()

# Predictions
y_pred = (y_prob > moving_threshold).astype(int)
# Remove NaN values (use NumPy masking)
valid_idx = ~np.isnan(moving_threshold)
y_true = y_test.to_numpy()[valid_idx]
y_pred = y_pred[valid_idx]
y_prob_valid = y_prob[valid_idx]

# Evaluation
print("===== Logistic Regression (Moving Threshold) =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))
print("AUC      :", roc_auc_score(y_true, y_prob_valid))


===== Logistic Regression (Moving Threshold) =====
Accuracy : 0.7232472324723247
Precision: 0.11392405063291139
Recall   : 0.6428571428571429
F1 Score : 0.1935483870967742
AUC      : 0.7565314063368538


### **Model Performance Comparison:**

| **Model**                                      | **Accuracy** | **Precision** | **Recall** | **F1 Score** | **AUC** |
| ---------------------------------------------- | ------------ | ------------- | ---------- | ------------ | ------- |
| **Logistic Regression (Class-Weight)**         | 0.77         | 0.14          | 0.625      | 0.22         | 0.77    |
| **Logistic Regression (Random Oversampling)**  | 0.78         | 0.14          | 0.625      | 0.23         | 0.77    |                                                                                                                
| **Logistic Regression (Random Undersampling)** | 0.69         | 0.11          | 0.69       | 0.19         | 0.76    |
| **Logistic Regression (SMOTE)**                | 0.77         | 0.14          | 0.63       | 0.22         | 0.77    |